In [1]:
import os
import numpy as np
import pandas as pd

current_dir = os.getcwd()
split_dir = os.path.join(current_dir, "centralized_split")
train_path = os.path.join(split_dir, "train.pkl")

df_train = pd.read_pickle(train_path)
print("Train shape:", df_train.shape)

feature_cols = ["input_ids", "attention_mask"]
label_cols = [c for c in df_train.columns if c not in feature_cols]
print("Num labels:", len(label_cols))
print("Labels:", label_cols)


Train shape: (9017, 14)
Num labels: 12
Labels: ['HS', 'Abusive', 'HS_Individual', 'HS_Group', 'HS_Religion', 'HS_Race', 'HS_Physical', 'HS_Gender', 'HS_Other', 'HS_Weak', 'HS_Moderate', 'HS_Strong']


In [2]:
def save_clients(client_dfs, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for cid, cdf in enumerate(client_dfs):
        p = os.path.join(out_dir, f"client_{cid:02d}.pkl")
        cdf.to_pickle(p)
    print(f"Saved {len(client_dfs)} clients to: {out_dir}")


In [3]:
NUM_CLIENTS = 10
rng = np.random.default_rng(42)

idx = np.arange(len(df_train))
rng.shuffle(idx)

splits = np.array_split(idx, NUM_CLIENTS)
client_dfs_iid = [df_train.iloc[s].reset_index(drop=True) for s in splits]

out_iid = os.path.join(current_dir, "fl_clients_iid_10")
save_clients(client_dfs_iid, out_iid)

# quick check
print([len(c) for c in client_dfs_iid])


Saved 10 clients to: c:\Users\Alif\OneDrive - uinjkt.ac.id\Kuliah UIN\Skripsi\eksperimen skripsi\fl_clients_iid_10
[902, 902, 902, 902, 902, 902, 902, 901, 901, 901]


In [4]:
NUM_CLIENTS = 10
ALPHA = 0.3
rng = np.random.default_rng(42)

Y = df_train[label_cols].values.astype(int)

# Untuk tiap sample, pilih "label dominan" = label pertama yang aktif.
# Jika semua nol (tidak ada label), masuk kelas tambahan "NONE".
dominant = []
none_class = len(label_cols)

for i in range(len(Y)):
    ones = np.where(Y[i] == 1)[0]
    dominant.append(int(ones[0]) if len(ones) > 0 else none_class)

dominant = np.array(dominant)
num_classes = none_class + 1  # label + NONE

# Kumpulkan index per kelas
class_indices = [np.where(dominant == k)[0] for k in range(num_classes)]
for k in range(num_classes):
    rng.shuffle(class_indices[k])

# Dirichlet proportions untuk tiap kelas -> dibagi ke client
client_indices = [[] for _ in range(NUM_CLIENTS)]

for k in range(num_classes):
    idx_k = class_indices[k]
    if len(idx_k) == 0:
        continue

    proportions = rng.dirichlet([ALPHA] * NUM_CLIENTS)
    # konversi proporsi menjadi jumlah index
    counts = (proportions * len(idx_k)).astype(int)

    # koreksi agar total tepat
    diff = len(idx_k) - counts.sum()
    for j in range(abs(diff)):
        counts[j % NUM_CLIENTS] += 1 if diff > 0 else -1

    start = 0
    for cid in range(NUM_CLIENTS):
        cnt = counts[cid]
        if cnt > 0:
            client_indices[cid].extend(idx_k[start:start+cnt].tolist())
        start += cnt

# Build client dfs
client_dfs_noniid = []
for cid in range(NUM_CLIENTS):
    c_idx = np.array(client_indices[cid], dtype=int)
    rng.shuffle(c_idx)
    cdf = df_train.iloc[c_idx].reset_index(drop=True)
    client_dfs_noniid.append(cdf)

out_noniid = os.path.join(current_dir, f"fl_clients_noniid_10_alpha{ALPHA}")
save_clients(client_dfs_noniid, out_noniid)

print([len(c) for c in client_dfs_noniid])


Saved 10 clients to: c:\Users\Alif\OneDrive - uinjkt.ac.id\Kuliah UIN\Skripsi\eksperimen skripsi\fl_clients_noniid_10_alpha0.3
[371, 271, 3331, 727, 1534, 1694, 371, 600, 21, 97]


In [5]:
def label_stats(df, label_cols):
    sums = df[label_cols].sum().sort_values(ascending=False)
    return sums

print("IID client 0 top labels:\n", label_stats(client_dfs_iid[0], label_cols).head(5))
print("\nNon-IID client 0 top labels:\n", label_stats(client_dfs_noniid[0], label_cols).head(5))


IID client 0 top labels:
 HS               349
Abusive          344
HS_Other         234
HS_Individual    229
HS_Weak          214
dtype: int64

Non-IID client 0 top labels:
 Abusive          46
HS                5
HS_Individual     5
HS_Weak           3
HS_Other          3
dtype: int64
